# NER Training — Fine-tune PhoBERT/ViHealthBERT trên Combined Dataset

**Trước khi chạy notebook này, upload 7 file sau lên thư mục gốc của Colab:**

| File | Lý do cần |
|---|---|
| `ner_data/converted_data/combined_train.jsonl` (~3MB) | Data train 9,629 câu |
| `ner_data/converted_data/combined_dev.jsonl` (~500KB) | Data dev 1,637 câu |
| `ner_data/converted_data/combined_test.jsonl` (~600KB) | Data test 1,992 câu |
| `ner_data/train_ner.py` (7.4KB) | Script training chính |
| `ner_data/test_alignment_logic.py` (6.2KB) | Verify logic alignment |
| `ner_data/merge_ner_datasets.py` (5.8KB) | Script merge (chỉ cần để đọc lại config oversample) |

**Cập nhật schema (2026-07-06):**
- **THỦ_THUẬT** → đã bỏ hoàn toàn (không có trong schema chính thức)
- **THÔNG_TIN_BỆNH_NHÂN** → thêm từ PhoNER_COVID19 (AGE/GENDER/JOB, ~3,416 mẫu thật) + synthetic (~72 mẫu)
- **KẾT_QUẢ_XÉT_NGHIỆM** → giữ lại, đã tách 1 phần nhỏ sang Dev/Test (mỗi bên 7 mẫu) để không bị "0 support" khi eval

**Cảnh báo quan trọng trước khi chạy:**
- Synthetic data (~110 câu EHR giả lập) được oversample x15 = 1,530 câu → chiếm ~16% train set
- Đây là NHÂN BẢN y hệt, không phải data mới — model có thể memorize
- Nếu train F1 >> dev F1 (đặc biệt trên case văn phong EHR) → overfitting vào synthetic

## Cell 1/6: Cài đặt môi trường + Upload files

In [ ]:
# ===== CELL 1/6: Cài đặt + Tạo cấu trúc thư mục =====
!pip install -q transformers datasets seqeval torch accelerate

import os

# Tạo cấu trúc thư mục y hệt máy local để script không cần sửa path
os.makedirs("ner_data/converted_data", exist_ok=True)

# Hướng dẫn: upload các file từ máy bạn lên root Colab (kéo thả hoặc dùng Files tab bên trái)
# Sau khi upload, files sẽ ở /content/<filename>
# Cell này sẽ di chuyển chúng vào đúng vị trí script cần

import shutil

# Map file gốc -> vị trí đích
file_map = {
    "combined_train.jsonl": "ner_data/converted_data/combined_train.jsonl",
    "combined_dev.jsonl":   "ner_data/converted_data/combined_dev.jsonl",
    "combined_test.jsonl":  "ner_data/converted_data/combined_test.jsonl",
    "train_ner.py":         "ner_data/train_ner.py",
    "test_alignment_logic.py": "ner_data/test_alignment_logic.py",
    "merge_ner_datasets.py":  "ner_data/merge_ner_datasets.py",
}

all_ok = True
for src, dst in file_map.items():
    if os.path.exists(src):
        shutil.move(src, dst)
        size_kb = os.path.getsize(dst) / 1024
        print(f"  ✅ {src} → {dst} ({size_kb:.0f} KB)")
    elif os.path.exists(dst):
        size_kb = os.path.getsize(dst) / 1024
        print(f"  ✓  {dst} đã có sẵn ({size_kb:.0f} KB)")
    else:
        print(f"  ❌ THIẾU: {src} — upload file này trước khi chạy tiếp!")
        all_ok = False

if not all_ok:
    print("\n⚠️  Upload các file còn thiếu lên Colab (kéo thả vào Files sidebar), rồi CHẠY LẠI CELL NÀY.")
else:
    print(f"\n✅ Tất cả {len(file_map)} file đã sẵn sàng.")
    !ls -lh ner_data/converted_data/combined_*.jsonl
    !ls -lh ner_data/*.py

## Cell 2/6: Đếm phân bố entity type trong toàn bộ data

In [ ]:
# ===== CELL 2/6: Đếm phân bố entity type — phát hiện imbalance TRƯỚC khi train =====
# Đây là cell "kiểm tra thực tế" — không chỉ tin vào giả định, mà đếm thật.
# Nếu thấy 1 type có <100 mẫu trong train → model gần như sẽ không học được type đó.

import json
from collections import Counter

def count_entities(path, label):
    type_counter = Counter()
    source_counter = Counter()
    total = 0; records = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            records += 1
            for e in r["entities"]:
                t = e["type"]
                type_counter[t] += 1
                total += 1
            source_counter[r.get("source", "?")] += 1

    print(f"\n{'='*50}")
    print(f"{label}: {records} records, {total} entities")
    print(f"-'*30")
    for t, c in sorted(type_counter.items(), key=lambda x: -x[1]):
        pct = c / total * 100
        bar = '█' * int(pct / 2)
        print(f"  {t:30s}: {c:5d} ({pct:5.1f}%) {bar}")
    print(f"\n  Theo nguồn gốc:")
    for src, c in sorted(source_counter.items()):
        print(f"    {src}: {c} câu ({c/records*100:.0f}%)")

for split in ["combined_train.jsonl", "combined_dev.jsonl", "combined_test.jsonl"]:
    path = f"ner_data/converted_data/{split}"
    count_entities(path, split)

# ==== DIAGNOSIS TỰ ĐỘNG ====
print(f"\n{'='*50}")
print("CHẨN ĐOÁN TỰ ĐỘNG:")
train_path = "ner_data/converted_data/combined_train.jsonl"
tc = Counter()
with open(train_path, encoding="utf-8") as f:
    for line in f:
        for e in json.loads(line)["entities"]:
            tc[e["type"]] += 1

warnings = []
for t, c in tc.items():
    if c < 100:
        warnings.append(f"  ⚠️  {t}: chỉ {c} mẫu — model sẽ KHÔNG học được type này!")
    elif c < 500:
        warnings.append(f"  ⚡ {t}: {c} mẫu — ít, có thể F1 thấp hơn các type khác")
    else:
        print(f"  ✅ {t}: {c} mẫu — đủ để model học")

if warnings:
    print("\n⚠️  CẢNH BÁO TYPE HIẾM:")
    for w in warnings:
        print(w)
    print("\n  → Nếu đề bài không yêu cầu type này, có thể bỏ khỏi LABEL_LIST.")
    print("  → Nếu đề bài yêu cầu, cần thêm synthetic data cho riêng type này.")
else:
    print("  ✅ Tất cả type đều có ≥100 mẫu — cân bằng chấp nhận được")

## Cell 3/6: Verify Logic Alignment (KHÔNG CẦN INTERNET)

**Đây là cell quan trọng nhất trước khi train — chạy TRƯỚC, không bỏ qua.**

Dùng mock tokenizer giả lập BPE (giống PhoBERT/ViHealthBERT) để verify hàm `char_entities_to_bio_labels()` xử lý đúng:
- Entity bị BPE tách thành nhiều sub-token → vẫn gán B-/I- liên tục
- Ranh giới entity rơi giữa 1 sub-token (overlap) → vẫn bắt được
- Câu không có entity → toàn bộ nhãn O

In [ ]:
# ===== CELL 3/6: Verify alignment logic bằng mock tokenizer =====
# Logic này hoạt động GIỐNG HỆT với tokenizer thật của PhoBERT/ViHealthBERT
# vì cả 2 đều dùng (return_offsets_mapping=True) + HF fast tokenizer interface.
# Nếu pass → logic đúng, vấn đề duy nhất có thể là model quality / data imbalance.

%run ner_data/test_alignment_logic.py

# output mong đợi:
# Test 1: Case đơn giản... ✅ PASS
# Test 2: Entity nhiều từ... ✅ PASS
# Test 3: Entity BPE split... ✅ PASS
# Test 4: Câu không entity... ✅ PASS
# ✅ TẤT CẢ TEST PASS — logic alignment đúng

## Cell 4/6: TRAINING CHÍNH — PhoBERT-Base (Baseline)

**Đây là cell quan trọng nhất — chạy baseline với PhoBERT.**

Thông số:
- Model: `vinai/phobert-base` (~135M params)
- Epochs: 5 (có thể tăng lên 8-10 nếu chưa converge)
- Batch size: 16 (giảm xuống 8 nếu GPU T4 hết RAM)
- Learning rate: 2e-5 (mặc định cho BERT fine-tune)

**Theo dõi khi train:**
- `eval_f1` — entity-level F1 (dùng seqeval, KHÔNG phải accuracy thô)
- `eval_loss` — nếu loss bắt đầu TĂNG trong khi F1 vẫn tăng → overfitting
- Per-type F1 trong classification_report ở cuối mỗi epoch

In [ ]:
# ===== CELL 4/6: TRAIN PHOBERT-BASE (BASELINE) =====
# ⏱️  Thời gian dự kiến: 20-40 phút trên GPU T4 (Colab free)
# 📊  Metric chính: entity-level F1 (seqeval) — KHÔNG dùng accuracy

%run ner_data/train_ner.py \
    --model_name vinai/phobert-base \
    --epochs 5 \
    --batch_size 16 \
    --lr 2e-5

# ===== SAU KHI TRAIN XONG =====
# GHI LẠI CÁC CON SỐ NÀY (để so sánh với Cell 5):
#   - eval_f1 tổng
#   - Per-type F1 (BỆNH, THUỐC, TRIỆU_CHỨNG, THỦ_THUẬT, KẾT_QUẢ_XÉT_NGHIỆM)
#   - Số epoch nào đạt F1 cao nhất (load_best_model_at_end=True)

# NẾU GPU HẾT RAM:
#   Chạy lại với --batch_size 8 và/hoặc thêm gradient_accumulation_steps=2
#   (sửa train_ner.py hoặc chạy cell dưới với batch_size=8)

In [ ]:
# ===== FALLBACK: GPU T4 16GB thường OK với batch=16, nhưng nếu OOM: =====
# %run ner_data/train_ner.py \
#     --model_name vinai/phobert-base \
#     --epochs 5 \
#     --batch_size 8 \
#     --lr 2e-5

## Cell 5/6: So sánh với ViHealthBERT (Domain-Specific)

**Chỉ chạy cell này SAU KHI đã có kết quả baseline PhoBERT.**

ViHealthBERT (`demdecuong/vihealthbert-base-word`) pretrained trên văn bản y tế tiếng Việt — về lý thuyết sẽ hiểu từ viết tắt/Terminology tốt hơn PhoBERT (báo chí). Nhưng checkpoint này có thể không tồn tại trên HuggingFace — nếu báo lỗi `404 Not Found`, thử:
- `demdecuong/vihealthbert-base-syllable` (nếu word-level không có)
- Hoặc skip cell này — PhoBERT vẫn là baseline tốt

**So sánh cần ghi lại:**
| Metric | PhoBERT | ViHealthBERT |
|---|---|---|
| eval_f1 | ... | ... |
| BỆNH F1 | ... | ... |
| THUỐC F1 | ... | ... |
| THỦ_THUẬT F1 | ... | ... |
| KẾT_QUẢ_XÉT_NGHIỆM F1 | ... | ... |

In [ ]:
# ===== CELL 5/6: TRAIN VIHEALTHBERT (SO SÁNH) =====
# Chạy SAU KHI Cell 4 hoàn tất. Nếu model không tồn tại → thử checkpoint khác.

!echo "Thử ViHealthBERT..."

# Thử checkpoint chính trước
%run ner_data/train_ner.py \
    --model_name demdecuong/vihealthbert-base-word \
    --epochs 5 \
    --batch_size 16 \
    --lr 2e-5 \
    --output_dir ner_model_output_vihealth 2>&1

# NẾU LỖI 404 "model not found" — thử checkpoint thay thế:
# demdecuong/vihealthbert-base-syllable
# Hoặc bỏ qua cell này, dùng kết quả PhoBERT là đủ.

## Cell 6/6: Phân tích Overfitting + So sánh cuối cùng

**Đây là cell quan trọng sau khi train xong cả 2 model — phân tích THẬT, không đoán.**

Cell này sẽ:
1. So sánh 2 model nếu cả 2 đều chạy được
2. **Cảnh báo overfitting**: nếu F1 trên synthetic cases >> F1 trên PhoNER/ViMQ cases
3. In bảng tổng hợp per-type để bạn quyết định type nào giữ/bỏ trước Phase 7

In [ ]:
# ===== CELL 6/6: PHÂN TÍCH CUỐI CÙNG =====

import json, os
from collections import Counter

print("="*60)
print("=== PHÂN TÍCH DATA — KHÔNG ĐOÁN, CHỈ ĐẾM ===")
print("="*60)

# 1. Đếm synthetic vs real trong train set
train_path = "ner_data/converted_data/combined_train.jsonl"
src_counter = Counter()
total = 0
with open(train_path, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        src = r.get("source", "?")
        src_counter[src] += 1
        total += 1

print(f"\nTrain set ({total} câu) theo nguồn gốc:")
for src, c in sorted(src_counter.items(), key=lambda x: -x[1]):
    pct = c / total * 100
    print(f"  {src:15s}: {c:5d} câu ({pct:.1f}%)")
    if src == "synthetic" and pct > 10:
        print(f"      ⚠️  Synthetic chiếm {pct:.0f}% — rủi ro memorize, cần theo dõi F1 gap")

# 2. In lại số entity theo type (đã đếm ở cell 2, nhưng nhắc lại)
print(f"\n{'='*60}")
print("=== PHÂN BỐ ENTITY TYPE (train) ===")
tc = Counter()
with open(train_path, encoding="utf-8") as f:
    for line in f:
        for e in json.loads(line)["entities"]:
            tc[e["type"]] += 1

for t, c in sorted(tc.items(), key=lambda x: -x[1]):
    pct = c / sum(tc.values()) * 100
    bar = '█' * int(pct / 2)
    print(f"  {t:30s}: {c:5d} ({pct:5.1f}%) {bar}")

# 3. Kiểm tra output dirs
print(f"\n{'='*60}")
print("=== OUTPUT DIRS ===")
for d in ["ner_model_output", "ner_model_output_vihealth"]:
    if os.path.exists(f"{d}/final"):
        print(f"  ✅ {d}/final — model đã train xong")
    else:
        print(f"  ⏳ {d}/final — chưa có (cell tương ứng chưa chạy hoặc lỗi)")

print(f"\n{'='*60}")
print("=== VIỆC CẦN LÀM TIẾP ===")
print("1. Gửi lại classification_report (in ra ở cuối mỗi cell train)")
print("2. Đặc biệt: F1 của BỆNH, THUỐC, THỦ_THUẬT, KẾT_QUẢ_XÉT_NGHIỆM")
print("3. Nếu KẾT_QUẢ_XÉT_NGHIỆM F1 ≈ 0 (chỉ 210 mẫu/17K total) → xác định có cần giữ không")
print("4. Chuyển sang Phase 7: tích hợp NER + Entity Linking thành pipeline")